# SmartLink — manual final recommendation review

Run the cells in order. Change the paths in **Cell 1** for your batch. No SmartLink package import or `pip install -e .` is needed.

For each subgroup touched by the input, use **all current subgroup members**. Each member votes for its input recommendation if present; otherwise it votes for its **current AMI FLOC**. These are meter counts, not confidence-weighted votes.

| Provisional distinct FLOCs | Your acceptance rule |
|---|---|
| 1 | Keep that assignment; export only actual changes |
| 2–3 | Leading FLOC must represent **strictly more than 40%** of all members |
| More than 3 | Leading FLOC must represent **at least 60%** of all members |

When a subgroup passes, assign its unique leading FLOC to **every member**. Tied leaders, missing members/assignments, and inconsistent subgroup membership go to review. There is no second `>50%` override after rejection.

Every member of an accepted subgroup must have the final transformer in its candidate file with a valid distance **≤550 ft**. One failure rejects the entire subgroup. Ungrouped input meters are checked individually. Missing candidate pairs mean **unverified eligibility**, not a proven distance greater than 550 ft.

`final_recommendations` contains only accepted changes relative to AMI, including changes added by subgroup enforcement. Rejected groups retain their current assignments. Connection Object conflicts and transformers with more than three subgroups are **review flags**, not new rejection rules. This notebook does not create COs or validate transformer phasing.

The input is assumed to be the recommendation set you already selected. No new score-gap threshold is applied. An original score gap is not a confidence estimate for a different transformer imposed by the subgroup.

## Cell 1 — paths and imports

Dependencies: `pandas`, `numpy`, `pyarrow`; `openpyxl` only if you enable Excel export.
If necessary, run `%pip install pandas numpy pyarrow openpyxl` in this notebook's kernel.

Use candidate files from the snapshot you intend to validate against. All division files needed by the selected recommendations must be available. Missing files stop the run rather than reject meters silently.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

BASE = Path(r"C:\Users\u355394\OneDrive - IBERDROLA S.A\Documents - Asset Data Strategy\Asset Data Quality\SmartLink")
RECOMMENDATIONS_PATH = BASE / "data/processed/recommendations/2026-10-23/all_recommendations.parquet"
AMI_PATH = BASE / "data/cache/ami/ny_ami.parquet"
ITRON_PATH = BASE / "data/cache/itron/itron_tx_subgroups.parquet"
CANDIDATE_PAIRS_DIR = BASE / "data/cache/candidates/2026-10-26/pairs"
OUTPUT_DIR = RECOMMENDATIONS_PATH.parent / "manual_final_review"

# None = all divisions in the input; ["Sodus"] = review that division only.
DIVISIONS = None
MAX_DISTANCE_FT = 550.0
EXPORT_EXCEL = False  # Parquet outputs are always written in the final cell.

# Canonical name -> actual AMI column. None tries the aliases in Cell 2.
# If a column is missing or ambiguous, the error lists the available columns.
AMI_COLUMN_MAP = {
    "meter_number": None,
    "division": None,
    "current_transformer": None,
    "connection_object": None,
    "premise_id": None,
}

## Cell 2 — small data helpers

Only needed columns are loaded. IDs are compared as strings; leading zeros are preserved if present in the source. Exact duplicate rows are removed, but conflicting rows for a meter stop the run. No arbitrary “keep first” is used.

In [ ]:
def clean_id(series):
    result = series.astype("string").str.strip()
    result = result.mask(result.str.lower().isin(["", "nan", "none", "null", "<na>"]))
    return result.str.replace(r"^(\d+)\.0$", r"\1", regex=True)


def unique_meters(frame, label):
    frame = frame.drop_duplicates().copy()
    if frame["meter_number"].isna().any():
        raise ValueError(f"{label}: blank meter_number(s).")
    duplicates = frame.loc[frame["meter_number"].duplicated(False), "meter_number"]
    if not duplicates.empty:
        raise ValueError(f"{label}: conflicting duplicate meters: {duplicates.head(10).tolist()}")
    return frame


def subgroup_key(ids, divisions):
    ids, divisions = clean_id(ids), clean_id(divisions).str.upper()
    tail = ids.str.rsplit("::", n=1).str[-1]
    result = (divisions + "::ITRON::" + tail).where(ids.notna())
    qualified = ids.str.contains("::", regex=False, na=False)
    bad = ids.notna() & (divisions.isna() | (qualified & ids.str.upper().ne(result.str.upper())))
    if bad.any():
        raise ValueError("Subgroup ID/division mismatch or missing division: "
                         + str(ids.loc[bad].head(10).tolist()))
    return result


def read_ami(path, column_map):
    aliases = {
        "meter_number": ["meter_number", "meter", "meter_id"],
        "division": ["division"],
        "current_transformer": ["current_transformer", "current_meter_floc", "meter_floc", "floc", "FuncLoc"],
        "connection_object": ["connection_object", "connection_object_id"],
        "premise_id": ["premise_id", "premise", "premiseID"],
    }
    available = pq.read_schema(path).names
    resolved = {}
    for target, options in aliases.items():
        explicit = column_map.get(target)
        matches = ([explicit] if explicit else
                   [c for c in available if c.casefold() in {a.casefold() for a in options}])
        if len(matches) != 1 or matches[0] not in available:
            raise ValueError(f"Set AMI_COLUMN_MAP[{target!r}] explicitly. Available columns: {available}")
        resolved[target] = matches[0]
    if len(set(resolved.values())) != len(resolved):
        raise ValueError("AMI_COLUMN_MAP must use a different source column for each field.")
    print("AMI columns:", resolved)
    return pd.read_parquet(path, columns=list(resolved.values())).rename(
        columns={source: target for target, source in resolved.items()}
    )

## Cell 3 — create the complete review population

AMI supplies current FLOCs, COs and premises. ITRON supplies current subgroup membership, keyed by **division + subgroup ID**, so identical numeric subgroup IDs in different divisions do not merge. AMI/ITRON FLOC differences and old model/AMI FLOC differences remain visible in the audit.

In [ ]:
def prepare_population(recommendations, ami, itron, divisions=None):
    required = {"meter_number", "division", "recommended_transformer", "subgroup_id"}
    missing = required - set(recommendations.columns)
    if missing:
        raise ValueError(f"Recommendations missing columns: {sorted(missing)}")
    ami = ami[["meter_number", "division", "current_transformer", "connection_object", "premise_id"]].copy()
    itron = itron[["meter_number", "Tx_Subgroup_ID", "current_meter_floc", "division"]].copy()
    keep = [c for c in ["meter_number", "division", "recommended_transformer", "subgroup_id",
                       "current_transformer", "score_gap", "recommended_distance_ft", "current_distance_ft"]
            if c in recommendations.columns]
    rec = recommendations[keep].copy()
    for frame in (ami, itron, rec):
        for column in frame.columns:
            if column not in {"score_gap", "recommended_distance_ft", "current_distance_ft"}:
                frame[column] = clean_id(frame[column])
        frame["division"] = frame["division"].str.upper()
    if divisions is not None:
        selected = {str(d).strip().upper() for d in divisions}
        rec = rec.loc[rec["division"].isin(selected)].copy()
    ami = unique_meters(ami, "AMI")
    itron = unique_meters(itron, "ITRON")
    rec = unique_meters(rec, "Recommendations")
    itron["subgroup_id"] = subgroup_key(itron["Tx_Subgroup_ID"], itron["division"])
    rec["subgroup_id"] = subgroup_key(rec["subgroup_id"], rec["division"])
    rec = rec.rename(columns={c: "model_" + c for c in rec if c != "meter_number"})
    rec["in_input_recommendations"] = True
    itron = itron.rename(columns={"division": "itron_division", "current_meter_floc": "itron_current_transformer"})
    population = ami.merge(itron.drop(columns="Tx_Subgroup_ID"), on="meter_number",
                           how="outer", validate="one_to_one", indicator="population_source")
    population["in_ami"] = population["population_source"].ne("right_only")
    population["division"] = population["division"].fillna(population["itron_division"])
    population = population.drop(columns="population_source")

    # Include both current subgroup members and any group named by an orphan input.
    input_meters = rec["meter_number"]
    affected = pd.Index(population.loc[population["meter_number"].isin(input_meters), "subgroup_id"].dropna())
    affected = affected.union(pd.Index(rec["model_subgroup_id"].dropna()))
    mask = population["meter_number"].isin(input_meters) | population["subgroup_id"].isin(affected)
    work = population.loc[mask].merge(rec, on="meter_number", how="outer", validate="one_to_one")
    work["in_ami"] = work["in_ami"].eq(True)
    work["in_input_recommendations"] = work["in_input_recommendations"].eq(True)
    work["division"] = work["division"].fillna(work["model_division"])
    cache_group = work["subgroup_id"].copy()
    work["subgroup_id"] = cache_group.fillna(work["model_subgroup_id"])
    model_group = work["model_subgroup_id"]
    work["subgroup_membership_mismatch"] = (
        model_group.notna() & (cache_group.isna() | model_group.ne(cache_group).fillna(False))
    )
    work["division_mismatch"] = (
        (work["itron_division"].notna() & work["division"].ne(work["itron_division"]).fillna(False)) |
        (work["model_division"].notna() & work["division"].ne(work["model_division"]).fillna(False))
    )
    for source in ("itron_current_transformer", "model_current_transformer"):
        if source not in work:
            work[source] = pd.NA
        work[source + "_differs_from_ami"] = (
            work[source].notna() & work["current_transformer"].notna()
            & work[source].ne(work["current_transformer"]).fillna(False)
        )
    work["provisional_transformer"] = work["model_recommended_transformer"].fillna(work["current_transformer"])
    work["invalid_member"] = (
        ~work["in_ami"] | work["division"].isna() | work["division_mismatch"]
        | work["subgroup_membership_mismatch"] | work["provisional_transformer"].isna()
        | (work["in_input_recommendations"] & work["model_recommended_transformer"].isna())
    )
    # Do not enforce a group that has no input meter after resolving current membership.
    active = work.loc[work["in_input_recommendations"], "subgroup_id"].dropna().unique()
    work = work.loc[work["in_input_recommendations"] | work["subgroup_id"].isin(active)].copy()
    return population, work.reset_index(drop=True)

## Cell 4 — your subgroup voting rules

Percentages use the entire subgroup size. They are captured **before** enforcement. Missing members, missing assignments or inconsistent membership reject the whole subgroup rather than shrink its denominator.

In [ ]:
def apply_group_rules(work):
    work = work.copy()
    members = work.loc[work["subgroup_id"].notna()]
    summary = members.groupby("subgroup_id", sort=False).agg(
        total_group_size=("meter_number", "size"),
        input_meter_count=("in_input_recommendations", "sum"),
        n_provisional_flocs=("provisional_transformer", "nunique"),
        invalid_member_count=("invalid_member", "sum"),
    )
    votes = members.groupby(["subgroup_id", "provisional_transformer"], sort=False).size().rename("votes").reset_index()
    votes["top_votes"] = votes.groupby("subgroup_id")["votes"].transform("max")
    leaders = votes.loc[votes["votes"].eq(votes["top_votes"])]
    winner = leaders.drop_duplicates("subgroup_id").set_index("subgroup_id")
    summary["majority_floc"] = winner["provisional_transformer"]
    summary["majority_meter_count"] = winner["votes"].reindex(summary.index).fillna(0).astype(int)
    summary["tied_leaders"] = leaders.groupby("subgroup_id").size().reindex(summary.index, fill_value=0).gt(1)
    summary["majority_percentage"] = summary["majority_meter_count"] / summary["total_group_size"] * 100
    summary.loc[summary["tied_leaders"], "majority_floc"] = pd.NA
    n, pct = summary["n_provisional_flocs"], summary["majority_percentage"]
    rule_pass = n.eq(1) | (n.isin([2, 3]) & pct.gt(40)) | (n.gt(3) & pct.ge(60))
    summary["decision"] = "ACCEPTED"
    summary.loc[~rule_pass, "decision"] = "REJECTED_VOTE_THRESHOLD"
    summary.loc[summary["tied_leaders"], "decision"] = "REJECTED_TIED_LEADERS"
    summary.loc[summary["invalid_member_count"].gt(0), "decision"] = "REJECTED_INCOMPLETE_OR_INCONSISTENT_DATA"

    work["decision"] = work["subgroup_id"].map(summary["decision"]).fillna("ACCEPTED")
    ungrouped_invalid = work["subgroup_id"].isna() & work["invalid_member"]
    work.loc[ungrouped_invalid, "decision"] = "REJECTED_INCOMPLETE_OR_INCONSISTENT_DATA"
    work["proposed_final_transformer"] = work["provisional_transformer"]
    enforce = work["subgroup_id"].notna() & work["decision"].eq("ACCEPTED")
    work.loc[enforce, "proposed_final_transformer"] = work.loc[enforce, "subgroup_id"].map(summary["majority_floc"])
    for field in ["total_group_size", "n_provisional_flocs", "majority_floc", "majority_percentage", "tied_leaders"]:
        work[field] = work["subgroup_id"].map(summary[field])
    return work, summary

## Cell 5 — validate every final candidate association

Reads one needed division file at a time, projecting only the three candidate columns. Checks **all** members, including members whose current transformer already equals the group's target. This preserves your whole-group rule. No transformer-coordinate schema is assumed and no existing model distance is reused for a changed target.

In [ ]:
def check_final_candidates(work, group_summary, pairs_dir, max_distance_ft=550.0):
    if not np.isfinite(max_distance_ft) or max_distance_ft < 0:
        raise ValueError("max_distance_ft must be finite and non-negative.")
    work, group_summary = work.copy(), group_summary.copy()
    pending = work.loc[work["decision"].eq("ACCEPTED"),
                       ["meter_number", "division", "proposed_final_transformer"]]
    paths = {}
    for path in Path(pairs_dir).glob("*.parquet"):
        key = path.stem.strip().upper()
        if key in paths:
            raise ValueError(f"Multiple candidate files for {key}.")
        paths[key] = path
    matches = []
    for division, wanted in pending.groupby("division", sort=False):
        if division not in paths:
            raise FileNotFoundError(f"Candidate file missing for {division}: {pairs_dir}")
        pairs = pd.read_parquet(paths[division], columns=["meter_number", "candidate_transformer", "distance_ft"])
        pairs["meter_number"] = clean_id(pairs["meter_number"])
        pairs["candidate_transformer"] = clean_id(pairs["candidate_transformer"])
        pairs = pairs.rename(columns={"candidate_transformer": "proposed_final_transformer",
                                      "distance_ft": "proposed_distance_ft"})
        found = wanted.drop(columns="division").merge(
            pairs, on=["meter_number", "proposed_final_transformer"], how="inner"
        ).drop_duplicates()
        if found["meter_number"].duplicated().any():
            raise ValueError(f"Conflicting candidate distances in {paths[division].name}.")
        found["candidate_pair_found"] = True
        matches.append(found)
        print(f"{division}: verified pair present for {len(found):,}/{len(wanted):,} members")
    if matches:
        found = pd.concat(matches, ignore_index=True)
        work = work.merge(found, on=["meter_number", "proposed_final_transformer"], how="left", validate="one_to_one")
    else:
        work["proposed_distance_ft"] = np.nan
        work["candidate_pair_found"] = False
    work["candidate_pair_found"] = work["candidate_pair_found"].eq(True)
    distance = pd.to_numeric(work["proposed_distance_ft"], errors="coerce")
    work["proposed_distance_ft"] = distance
    checked = work["decision"].eq("ACCEPTED")
    valid = work["candidate_pair_found"] & np.isfinite(distance) & distance.between(0, max_distance_ft)
    work["Distance_within_550_Feet"] = valid.astype("boolean").where(checked)
    work["candidate_check_reason"] = np.select(
        [~checked, ~work["candidate_pair_found"], ~np.isfinite(distance) | distance.lt(0), distance.gt(max_distance_ft)],
        ["NOT_CHECKED_VOTE_OR_DATA_REJECTED", "PAIR_NOT_IN_CANDIDATE_FILE", "INVALID_CANDIDATE_DISTANCE", "DISTANCE_EXCEEDS_LIMIT"],
        default="PASS",
    )
    failed = checked & ~valid
    failed_groups = work.loc[failed, "subgroup_id"].dropna().unique()
    group_summary["candidate_checks_failed"] = (
        work.loc[failed].groupby("subgroup_id").size().reindex(group_summary.index, fill_value=0)
    )
    group_summary.loc[group_summary.index.isin(failed_groups), "decision"] = "REJECTED_CANDIDATE_CHECK"
    work.loc[work["subgroup_id"].isin(failed_groups) | (failed & work["subgroup_id"].isna()), "decision"] = "REJECTED_CANDIDATE_CHECK"
    work["Okay_to_recommend"] = work["decision"].eq("ACCEPTED")
    work["Final_Recommended_FLOC"] = work["current_transformer"].where(
        ~work["Okay_to_recommend"], work["proposed_final_transformer"]
    )
    changed = ~work["Final_Recommended_FLOC"].eq(work["current_transformer"]).fillna(False)
    work["is_change"] = work["Okay_to_recommend"] & work["Final_Recommended_FLOC"].notna() & changed
    work["added_by_subgroup"] = work["is_change"] & ~work["in_input_recommendations"]
    work["final_matches_model"] = (work["in_input_recommendations"]
                                   & work["Final_Recommended_FLOC"].eq(work["model_recommended_transformer"]).fillna(False))
    work["recommended_distance_ft"] = distance.where(work["Okay_to_recommend"])
    return work, group_summary.reset_index()

## Cell 6 — build the change set and whole-AMI review tables

CO checks include **every AMI meter in each affected CO**, including meters outside the recommendation input or the affected subgroups. Conflicts do not silently remove a few members from an accepted subgroup. `connection_object_review_required=True` means the transformer change needs a separate CO decision before SAP updating; no new CO IDs are generated here.

`transformer_subgroup_review` counts all current subgroup associations after applying accepted changes, restricted to target transformers receiving a change. More than three is flagged for review. Phasing is not validated because no phase mapping/rule was supplied.

In [ ]:
def build_outputs(population, audit, group_summary):
    final = audit.loc[audit["is_change"]].copy()
    assignments = final.set_index("meter_number")["Final_Recommended_FLOC"]
    state = population.loc[population["in_ami"],
                           ["meter_number", "division", "connection_object", "premise_id", "current_transformer", "subgroup_id"]].copy()
    state["Final_Recommended_FLOC"] = state["meter_number"].map(assignments).fillna(state["current_transformer"])
    affected_cos = final["connection_object"].dropna().unique()
    co_members = state.loc[state["connection_object"].isin(affected_cos)].copy()
    co_members["is_change"] = co_members["meter_number"].isin(assignments.index)
    co_members["missing_final_assignment"] = co_members["Final_Recommended_FLOC"].isna()
    co_summary = co_members.groupby("connection_object", sort=False).agg(
        total_ami_meters=("meter_number", "size"),
        current_transformer_count=("current_transformer", "nunique"),
        final_transformer_count=("Final_Recommended_FLOC", "nunique"),
        changed_meters=("is_change", "sum"),
        missing_final_assignments=("missing_final_assignment", "sum"),
    ).reset_index()
    co_summary["connection_object_review_required"] = (
        co_summary["final_transformer_count"].gt(1) | co_summary["missing_final_assignments"].gt(0)
    )
    co_flags = co_summary.set_index("connection_object")["connection_object_review_required"]
    for frame in (final, audit):
        frame["connection_object_review_required"] = frame["connection_object"].map(co_flags).astype("boolean")
        frame.loc[frame["connection_object"].isna(), "connection_object_review_required"] = True
    final["recommended_transformer"] = final["Final_Recommended_FLOC"]

    tx_counts = (state.dropna(subset=["subgroup_id", "Final_Recommended_FLOC"])
                 .groupby("Final_Recommended_FLOC")["subgroup_id"].nunique())
    tx_review = tx_counts.reindex(final["Final_Recommended_FLOC"].unique(), fill_value=0).rename("subgroup_count").reset_index()
    tx_review["more_than_3_subgroups"] = tx_review["subgroup_count"].gt(3)
    final["transformer_subgroup_count"] = final["Final_Recommended_FLOC"].map(tx_counts).fillna(0).astype(int)
    final["transformer_has_more_than_3_subgroups"] = final["transformer_subgroup_count"].gt(3)
    leading = ["meter_number", "division", "connection_object", "premise_id", "subgroup_id",
               "current_transformer", "recommended_transformer", "Final_Recommended_FLOC",
               "recommended_distance_ft", "in_input_recommendations", "added_by_subgroup",
               "model_recommended_transformer", "model_score_gap", "final_matches_model",
               "total_group_size", "n_provisional_flocs", "majority_percentage",
               "connection_object_review_required", "transformer_subgroup_count",
               "transformer_has_more_than_3_subgroups", "model_current_distance_ft"]
    # The compact change file keeps useful review fields; full diagnostics stay in the audit.
    final = final[[c for c in leading if c in final]]
    return {
        "final_recommendations": final.reset_index(drop=True),
        "all_member_audit": audit.reset_index(drop=True),
        "group_summary": group_summary,
        "rejected_members": audit.loc[~audit["Okay_to_recommend"]].reset_index(drop=True),
        "connection_object_summary": co_summary,
        "connection_object_members": co_members.reset_index(drop=True),
        "transformer_subgroup_review": tx_review,
    }

## Cell 7 — load your files and run

All helpers accept DataFrames, so you can also pass an already filtered `df_recommendations` to `prepare_population`. Accepted groups have one final transformer; rejected groups can retain their pre-existing multiple transformers because no changes were approved.

In [ ]:
rec_columns = ["meter_number", "division", "connection_object", "premise_id", "current_transformer",
               "recommended_transformer", "subgroup_id", "recommended_distance_ft", "current_distance_ft", "score_gap"]
available = pq.read_schema(RECOMMENDATIONS_PATH).names
df_recommendations = pd.read_parquet(RECOMMENDATIONS_PATH, columns=[c for c in rec_columns if c in available])
df_ami = read_ami(AMI_PATH, AMI_COLUMN_MAP)
itron_today = pd.read_parquet(ITRON_PATH, columns=["meter_number", "Tx_Subgroup_ID", "current_meter_floc", "division"])

population, member_audit = prepare_population(df_recommendations, df_ami, itron_today, DIVISIONS)
member_audit, subgroup_summary = apply_group_rules(member_audit)
member_audit, subgroup_summary = check_final_candidates(member_audit, subgroup_summary, CANDIDATE_PAIRS_DIR, MAX_DISTANCE_FT)
results = build_outputs(population, member_audit, subgroup_summary)

final_recommendations = results["final_recommendations"]
group_summary = results["group_summary"]
all_member_audit = results["all_member_audit"]
rejected_members = results["rejected_members"]

# Invariants on the actual result, not just the input recommendations.
accepted_members = all_member_audit.loc[all_member_audit["Okay_to_recommend"]]
assert accepted_members.groupby("subgroup_id")["Final_Recommended_FLOC"].nunique().le(1).all()
assert final_recommendations["meter_number"].is_unique
assert final_recommendations["recommended_distance_ft"].between(0, MAX_DISTANCE_FT).all()
assert (~final_recommendations["current_transformer"].eq(final_recommendations["Final_Recommended_FLOC"]).fillna(False)).all()

display(pd.Series({
    "input_meters_reviewed": int(all_member_audit["in_input_recommendations"].sum()),
    "all_members_reviewed": len(all_member_audit),
    "accepted_subgroups": int(group_summary["decision"].eq("ACCEPTED").sum()),
    "rejected_subgroups": int(group_summary["decision"].ne("ACCEPTED").sum()),
    "final_meter_changes": len(final_recommendations),
    "changes_added_by_subgroup": int(final_recommendations["added_by_subgroup"].sum()),
    "changes_needing_CO_review": int(final_recommendations["connection_object_review_required"].sum()),
}, name="count").to_frame())
display(group_summary.head(20))
display(final_recommendations.head(20))

## Cell 8 — inspect a subgroup

The first table shows pre-enforcement proportions and the whole-group decision. The second shows each member's current, model, provisional, proposed and applied assignment, including members whose assignments stay unchanged.

In [ ]:
SUBGROUP_TO_REVIEW = "SODUS::ITRON::303202"
display(group_summary.loc[group_summary["subgroup_id"].eq(SUBGROUP_TO_REVIEW)])
review_columns = ["meter_number", "current_transformer", "model_recommended_transformer",
                  "provisional_transformer", "proposed_final_transformer", "Final_Recommended_FLOC",
                  "in_input_recommendations", "added_by_subgroup", "proposed_distance_ft",
                  "candidate_check_reason", "decision", "connection_object_review_required"]
display(all_member_audit.loc[all_member_audit["subgroup_id"].eq(SUBGROUP_TO_REVIEW), review_columns])

## Cell 9 — export

`final_recommendations.parquet` is the accepted **change set**. `all_member_audit.parquet` includes unchanged and rejected reviewed members; it is not a full-AMI export. `connection_object_members.parquet` includes all AMI members of affected COs.

Exports use the fixed output subfolder in Cell 1 and replace files with the same names when rerun. The original input and caches are not modified. The JSON records the input paths and thresholds. Optional Excel sheets are split at Excel's row limit.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, table in results.items():
    table.to_parquet(OUTPUT_DIR / f"{name}.parquet", index=False)

settings = {
    "recommendations": str(RECOMMENDATIONS_PATH), "ami": str(AMI_PATH), "itron": str(ITRON_PATH),
    "candidate_pairs_directory": str(CANDIDATE_PAIRS_DIR), "divisions": DIVISIONS,
    "max_distance_ft": MAX_DISTANCE_FT, "two_or_three_flocs_threshold": ">40%",
    "more_than_three_flocs_threshold": ">=60%", "ties": "reject subgroup",
    "confidence": "input already selected; no additional score-gap filter",
    "connection_object_conflicts": "flag for manual review; no automatic CO creation",
    "transformer_phasing": "not checked",
}
(OUTPUT_DIR / "review_settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")

if EXPORT_EXCEL:
    excel_tables = {
        "Final Recommendations": final_recommendations,
        "Group Summary": group_summary,
        "CO Summary": results["connection_object_summary"],
        "Transformer Subgroups": results["transformer_subgroup_review"],
    }
    with pd.ExcelWriter(OUTPUT_DIR / "manual_final_review.xlsx", engine="openpyxl") as writer:
        for name, table in excel_tables.items():
            for part, start in enumerate(range(0, max(1, len(table)), 1_048_575), start=1):
                table.iloc[start:start + 1_048_575].to_excel(writer, sheet_name=f"{name[:27]} {part}", index=False)

print(f"Saved {len(final_recommendations):,} meter changes and review tables to {OUTPUT_DIR}")